# Danish recognizer fine-tune + full card-crop prediction

Two-stage recipe (frozen-backbone warm-up, then full unfreeze at low LR)
reused from the earlier, now-superseded word-crop fine-tune attempt, which
converged with this exact shape. Attach both input datasets before running:
`museumscat-danish-recognizer-word-boxes` and `museum-card-crops-train-test`.
See docs/superpowers/specs/2026-07-29-card-word-boxes-danish-finetune-design.md.

## Cell 1: install doctr and fetch the pinned vendored training script

In [ ]:
!pip install -q "python-doctr[torch]==1.0.1"
!mkdir -p doctr_recognition
!curl -s -o doctr_recognition/train.py "https://raw.githubusercontent.com/mindee/doctr/v1.0.1/references/recognition/train.py"
!curl -s -o doctr_recognition/utils.py "https://raw.githubusercontent.com/mindee/doctr/v1.0.1/references/recognition/utils.py"


## Cell 2: locate the attached word-box dataset's train/val folders

In [ ]:
import glob
import os

def _find_dataset_dir(slug):
    candidates = glob.glob(f"/kaggle/input/**/{slug}", recursive=True)
    for c in candidates:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError(f"dataset '{slug}' not found under /kaggle/input (checked: {candidates})")

INPUT_DIR = _find_dataset_dir("museumscat-danish-recognizer-word-boxes")
TRAIN_PATH = os.path.join(INPUT_DIR, "train")
VAL_PATH = os.path.join(INPUT_DIR, "val")
assert os.path.isfile(os.path.join(TRAIN_PATH, "labels.json"))
assert os.path.isfile(os.path.join(VAL_PATH, "labels.json"))
print("train:", TRAIN_PATH)
print("val:", VAL_PATH)


## Cell 3: Stage 1 -- frozen backbone, train the reinitialized head

In [ ]:
!cd doctr_recognition && python train.py crnn_vgg16_bn \
    --vocab danish --pretrained --freeze-backbone --amp \
    --train_path {TRAIN_PATH} --val_path {VAL_PATH} \
    --epochs 1500 --batch_size 16 --lr 0.001 \
    --early-stop --early-stop-epochs 120 --early-stop-delta 0.0005 \
    --output_dir /kaggle/working --name danish_crnn_vgg16_bn_stage1


## Cell 4: confirm Stage 1's checkpoint was written

In [ ]:
import os
assert os.path.isfile("/kaggle/working/danish_crnn_vgg16_bn_stage1.pt")
print("stage 1 checkpoint ready")


## Cell 5: Stage 2 -- unfreeze the whole network, fine-tune at a low LR from Stage 1's weights

In [ ]:
!cd doctr_recognition && python train.py crnn_vgg16_bn \
    --vocab danish --resume /kaggle/working/danish_crnn_vgg16_bn_stage1.pt --amp \
    --train_path {TRAIN_PATH} --val_path {VAL_PATH} \
    --epochs 800 --batch_size 16 --lr 0.0001 --optim adamw --wd 0.0001 \
    --early-stop --early-stop-epochs 100 --early-stop-delta 0.0005 \
    --output_dir /kaggle/working --name danish_crnn_vgg16_bn_stage2


## Cell 6: confirm the final checkpoint was written

In [ ]:
import os
assert os.path.isfile("/kaggle/working/danish_crnn_vgg16_bn_stage2.pt")
print("final checkpoint ready")


## Cell 7: locate the already-uploaded card-crops dataset (images + manifest)

In [ ]:
CARDS_INPUT_DIR = _find_dataset_dir("museum-card-crops-train-test")
CARDS_IMAGES_DIR = os.path.join(CARDS_INPUT_DIR, "images")
MANIFEST_PATH = os.path.join(CARDS_INPUT_DIR, "manifest.csv")
assert os.path.isfile(MANIFEST_PATH)
print("manifest:", MANIFEST_PATH)
os.environ["MANIFEST_PATH"] = MANIFEST_PATH
os.environ["CARDS_IMAGES_DIR"] = CARDS_IMAGES_DIR


## Cell 8: write the per-shard prediction script (one process per GPU)

In [ ]:
with open("/kaggle/working/predict_shard.py", "w") as f:
    f.write('import argparse\nimport os\n\nimport cv2\nimport pandas as pd\nimport torch\nfrom doctr.datasets import VOCABS\nfrom doctr.models import ocr_predictor, recognition\n\nparser = argparse.ArgumentParser()\nparser.add_argument("--shard", type=int, required=True)\nparser.add_argument("--num-shards", type=int, required=True)\nparser.add_argument("--checkpoint", type=str, required=True)\nparser.add_argument("--manifest", type=str, required=True)\nparser.add_argument("--images-dir", type=str, required=True)\nparser.add_argument("--output", type=str, required=True)\nargs = parser.parse_args()\n\nreco_model = recognition.__dict__["crnn_vgg16_bn"](pretrained=False, vocab=VOCABS["danish"])\nreco_model.load_state_dict(torch.load(args.checkpoint, map_location="cpu", weights_only=True))\npredictor = ocr_predictor(\n    det_arch="db_resnet50", reco_arch=reco_model, pretrained=True, assume_straight_pages=True\n).cuda()\n\nmanifest = pd.read_csv(args.manifest)\ncrop_filenames = manifest["crop_filename"].tolist()[args.shard::args.num_shards]\n\nrows = []\nfor crop_filename in crop_filenames:\n    img_bgr = cv2.imread(os.path.join(args.images_dir, crop_filename))\n    if img_bgr is None:\n        continue\n    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)\n    result = predictor([img_rgb])\n    for page in result.pages:\n        for block in page.blocks:\n            for line in block.lines:\n                for word in line.words:\n                    rows.append({\n                        "crop_filename": crop_filename,\n                        "predicted_text": word.value,\n                        "confidence": word.confidence,\n                    })\n\npd.DataFrame(rows, columns=["crop_filename", "predicted_text", "confidence"]).to_csv(args.output, index=False)\nprint(f"shard {args.shard}: wrote {len(rows)} rows for {len(crop_filenames)} cards")\n')
print("wrote /kaggle/working/predict_shard.py")


## Cell 8b: warm the doctr detector weights cache once, before the two parallel shard processes start (avoids a race on the shared download cache)

In [ ]:
from doctr.models import ocr_predictor
_ = ocr_predictor(det_arch="db_resnet50", pretrained=True, assume_straight_pages=True)
print("detector weights cached")


## Cell 9: run the fine-tuned recognizer over every card crop, splitting the manifest across both GPUs (2 parallel processes, CUDA_VISIBLE_DEVICES 0/1)

In [ ]:
%%bash
cd /kaggle/working
CUDA_VISIBLE_DEVICES=0 python predict_shard.py --shard 0 --num-shards 2 \
    --checkpoint /kaggle/working/danish_crnn_vgg16_bn_stage2.pt \
    --manifest "$MANIFEST_PATH" --images-dir "$CARDS_IMAGES_DIR" \
    --output /kaggle/working/predictions_shard0.csv > shard0.log 2>&1 &
CUDA_VISIBLE_DEVICES=1 python predict_shard.py --shard 1 --num-shards 2 \
    --checkpoint /kaggle/working/danish_crnn_vgg16_bn_stage2.pt \
    --manifest "$MANIFEST_PATH" --images-dir "$CARDS_IMAGES_DIR" \
    --output /kaggle/working/predictions_shard1.csv > shard1.log 2>&1 &
wait
cat shard0.log shard1.log


## Cell 10: merge the two shard CSVs into predictions.csv

In [ ]:
import pandas as pd

shard0 = pd.read_csv("/kaggle/working/predictions_shard0.csv")
shard1 = pd.read_csv("/kaggle/working/predictions_shard1.csv")
pd.concat([shard0, shard1], ignore_index=True).to_csv("/kaggle/working/predictions.csv", index=False)
print(f"merged {len(shard0)} + {len(shard1)} = {len(shard0) + len(shard1)} rows")


## Cell 11: confirm predictions.csv was written

In [ ]:
import os
assert os.path.isfile("/kaggle/working/predictions.csv")
print("predictions ready: /kaggle/working/predictions.csv")
